VII. Regularization and Shrinkage (5 points)
What you should do:


- Compare at least two models (e.g., simple vs regularized)
- Explain how shrinkage affects performance
Connection to class: Bias–variance + Regularization

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score


df = pd.read_csv("paddydataset.csv")

# strip accidental spaces from column names
df.columns = df.columns.str.strip()

target = "Paddy yield(in Kg)"
X = df.drop(columns=[target])
y = df[target]

# identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Simple Model

In [ ]:
ols_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

ols_model.fit(X_train, y_train)

y_train_pred_ols = ols_model.predict(X_train)
y_test_pred_ols = ols_model.predict(X_test)

ols_train_mse = mean_squared_error(y_train, y_train_pred_ols)
ols_test_mse = mean_squared_error(y_test, y_test_pred_ols)
ols_train_r2 = r2_score(y_train, y_train_pred_ols)
ols_test_r2 = r2_score(y_test, y_test_pred_ols)

##Regularized model

In [ ]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100, 1000]
}

ridge_cv = GridSearchCV(
    ridge_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error"
)

ridge_cv.fit(X_train, y_train)

best_ridge = ridge_cv.best_estimator_

y_train_pred_ridge = best_ridge.predict(X_train)
y_test_pred_ridge = best_ridge.predict(X_test)

ridge_train_mse = mean_squared_error(y_train, y_train_pred_ridge)
ridge_test_mse = mean_squared_error(y_test, y_test_pred_ridge)
ridge_train_r2 = r2_score(y_train, y_train_pred_ridge)
ridge_test_r2 = r2_score(y_test, y_test_pred_ridge)

##How shrinkage affects performance

In [ ]:
print("OLS Results")
print(f"Train MSE: {ols_train_mse:.2f}")
print(f"Test MSE:  {ols_test_mse:.2f}")
print(f"Train R^2: {ols_train_r2:.4f}")
print(f"Test R^2:  {ols_test_r2:.4f}")
print()

print("Ridge Results")
print(f"Best alpha: {ridge_cv.best_params_['model__alpha']}")
print(f"Train MSE: {ridge_train_mse:.2f}")
print(f"Test MSE:  {ridge_test_mse:.2f}")
print(f"Train R^2: {ridge_train_r2:.4f}")
print(f"Test R^2:  {ridge_test_r2:.4f}")

OLS Results
Train MSE: 872263.25
Test MSE:  861065.18
Train R^2: 0.9898
Test R^2:  0.9894

Ridge Results
Best alpha: 0.1
Train MSE: 872264.77
Test MSE:  861120.75
Train R^2: 0.9898
Test R^2:  0.9894


Shrinkage is most useful when a model is overfitting or when predictors are highly correlated. In this dataset, since OLS already generalizes well, regularization provides little benefit, demonstrating that the effectiveness of shrinkage depends on the underlying data structure.